# Pokémon Wordle: Member 1 (Data and game engine)
Run the cells **top to bottom**. Before starting: in Google Drive create a folder called `pokemon_wordle` inside **My Drive** and share it with your 4 teammates as **Editors**.

Steps in this notebook:
1. Mount Drive
2. Create subfolders and stub files for teammates (tell the group when done)
3. Save a 20-row sample CSV, `loader.py`, `engine.py`, and run the tests (tell the group when done)
4. Download the full 386 Pokémon from PokéAPI, check them, replace the sample (tell the group when done)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
DRIVE = '/content/drive/MyDrive/pokemon_wordle'
sys.path.append(DRIVE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Subfolders, README and stub files for everyone
Safe to re-run: it never overwrites an existing file.

In [ ]:
import os

for folder in ["data", "results", "notebooks", "slides", "report"]:
    os.makedirs(f"{DRIVE}/{folder}", exist_ok=True)

STUBS = {
"basic_solvers.py": '''"""basic_solvers.py (owner: Member 2)
Solver interface: solver(pokemon, history) -> a Pokemon record
history is a list of (guess_record, clue). Never repeat a guess.
"""
from engine import feedback


def random_solver(pokemon, history):
    raise NotImplementedError


def filter_candidates(pokemon, history):
    """Keep a Pokemon only if it would give every clue seen so far and was not guessed yet."""
    raise NotImplementedError


def filter_solver(pokemon, history):
    raise NotImplementedError
''',
"entropy_solver.py": '''"""entropy_solver.py (owner: Member 3)"""
from engine import feedback


def entropy(guess, candidates):
    raise NotImplementedError


def entropy_solver(pokemon, history):
    raise NotImplementedError
''',
"minimax_solver.py": '''"""minimax_solver.py (owner: Member 4)"""
from engine import feedback


def minimax_solver(pokemon, history):
    raise NotImplementedError
''',
"benchmark.py": '''"""benchmark.py (owner: Member 4)"""
from engine import play_game, FAIL_SCORE


def run_benchmark():
    raise NotImplementedError
''',
"app.py": '''"""app.py (owner: Member 5)"""


def run_app():
    raise NotImplementedError
''',
"README.txt": """Pokemon Wordle (AI group project)
How to run:
1. In Colab start every notebook with the setup cell (mount Drive, sys.path.append).
2. Code files are in the main folder. Only edit your own files.
3. Interface: solver(pokemon, history) -> Pokemon record. history = list of (guess, clue).
4. After a teammate changes a file: Runtime > Restart session, then rerun your setup cell.
Owners: engine/loader/data = M1, basic_solvers = M2, entropy = M3, minimax/benchmark = M4, app/main/slides = M5
""",
}

for name, content in STUBS.items():
    path = f"{DRIVE}/{name}"
    if not os.path.exists(path):            # never overwrite a teammate's work
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        print("created", name)
    else:
        print("skipped (already exists)", name)

print(os.listdir(DRIVE))

created basic_solvers.py
created entropy_solver.py
created minimax_solver.py
created benchmark.py
created app.py
created README.txt
['data', 'results', 'notebooks', 'slides', 'report', 'basic_solvers.py', 'entropy_solver.py', 'minimax_solver.py', 'benchmark.py', 'app.py', 'README.txt']


## 3a. Sample CSV (20 Pokémon, so others can start testing)

In [ ]:
%%writefile /content/drive/MyDrive/pokemon_wordle/data/pokemon.csv
name,gen,type1,type2,height,weight
Bulbasaur,1,Grass,Poison,0.7,6.9
Charmander,1,Fire,None,0.6,8.5
Squirtle,1,Water,None,0.5,9.0
Charizard,1,Fire,Flying,1.7,90.5
Pikachu,1,Electric,None,0.4,6.0
Gyarados,1,Water,Flying,6.5,235.0
Geodude,1,Rock,Ground,0.4,20.0
Gengar,1,Ghost,Poison,1.5,40.5
Snorlax,1,Normal,None,2.1,460.0
Dragonite,1,Dragon,Flying,2.2,210.0
Mewtwo,1,Psychic,None,2.0,122.0
Togepi,2,Fairy,None,0.3,1.5
Marill,2,Water,Fairy,0.4,8.5
Lugia,2,Psychic,Flying,5.2,216.0
Tyranitar,2,Rock,Dark,2.0,202.0
Treecko,3,Grass,None,0.5,5.0
Blaziken,3,Fire,Fighting,1.9,52.0
Gardevoir,3,Psychic,Fairy,1.6,48.4
Rayquaza,3,Dragon,Flying,7.0,206.5
Mudkip,3,Water,None,0.4,7.6

Writing /content/drive/MyDrive/pokemon_wordle/data/pokemon.csv


## 3b. loader.py

In [ ]:
%%writefile /content/drive/MyDrive/pokemon_wordle/loader.py
import csv
import os


def load_pokemon(path=None):
    """Read the CSV and return a list of Pokemon records (dicts).

    Each record has exactly six fields:
      name (str), gen (int), type1 (str), type2 (str, "None" if missing),
      height (float, metres), weight (float, kg)
    """
    if path is None:
        here = os.path.dirname(os.path.abspath(__file__))
        path = os.path.join(here, "data", "pokemon.csv")
    records = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            records.append({
                "name": row["name"],
                "gen": int(row["gen"]),
                "type1": row["type1"],
                "type2": row["type2"],
                "height": float(row["height"]),
                "weight": float(row["weight"]),
            })
    return records

Writing /content/drive/MyDrive/pokemon_wordle/loader.py


## 3c. engine.py (clue rules, Game class, play_game)
Ask the team to confirm the 10% band before you finalise this.

In [ ]:
%%writefile /content/drive/MyDrive/pokemon_wordle/engine.py
"""engine.py  (owner: Member 1)

ALL clue rules live ONLY in this file.

A Pokemon record is a dict: name, gen, type1, type2, height, weight.
A clue is a tuple of 5 values: (generation, type1, type2, height, weight)
  generation / height / weight -> "green", "up" or "down"
        green = same generation / within 10% band
        up    = the answer is higher than the guess
        down  = the answer is lower than the guess
  type1 / type2 -> "green", "amber" or "grey"
History is a list of (guess_record, clue) pairs.
Solver interface (everyone uses this):  solver(pokemon, history) -> a Pokemon record
"""
import random

MAX_GUESSES = 8      # guesses allowed per game
FAIL_SCORE = 9       # a lost game counts as 9 guesses in the benchmark
BAND = 0.10          # 10% band for height and weight


def _number_clue(guess_value, answer_value):
    """green if the answer is within BAND of the guess, else up/down."""
    if abs(answer_value - guess_value) <= BAND * guess_value + 1e-9:
        return "green"
    return "up" if answer_value > guess_value else "down"


def feedback(guess, answer):
    """Compare a guessed Pokemon with the secret one and return the clue tuple."""
    # Generation
    if guess["gen"] == answer["gen"]:
        gen = "green"
    else:
        gen = "up" if answer["gen"] > guess["gen"] else "down"

    # Type 1: green = same slot, amber = type is in the answer's other slot
    if guess["type1"] == answer["type1"]:
        t1 = "green"
    elif guess["type1"] == answer["type2"]:
        t1 = "amber"
    else:
        t1 = "grey"

    # Type 2: None counts as a normal value for green; never amber for None
    if guess["type2"] == answer["type2"]:
        t2 = "green"
    elif guess["type2"] != "None" and guess["type2"] == answer["type1"]:
        t2 = "amber"
    else:
        t2 = "grey"

    height = _number_clue(guess["height"], answer["height"])
    weight = _number_clue(guess["weight"], answer["weight"])
    return (gen, t1, t2, height, weight)


class Game:
    """One game: a secret Pokemon, a history of guesses, win/lose after 8 guesses."""

    def __init__(self, pokemon, secret=None, max_guesses=MAX_GUESSES, seed=None):
        self.pokemon = pokemon
        self.secret = secret if secret is not None else random.Random(seed).choice(pokemon)
        self.max_guesses = max_guesses
        self.history = []          # list of (guess_record, clue)
        self.status = "playing"    # "playing", "won" or "lost"

    def guess(self, guess_record):
        """Make one guess, return the clue. Winning = guessing the secret by name
        (not 'all green', because the 10% band lets different Pokemon look alike)."""
        if self.status != "playing":
            raise ValueError("Game is already over")
        clue = feedback(guess_record, self.secret)
        self.history.append((guess_record, clue))
        if guess_record["name"] == self.secret["name"]:
            self.status = "won"
        elif len(self.history) >= self.max_guesses:
            self.status = "lost"
        return clue

    def guesses_used(self):
        return len(self.history)


def play_game(solver, pokemon, secret):
    """Let a solver play a full game against a given secret Pokemon.
    Returns (score, game). score = guesses used, or FAIL_SCORE (9) if lost."""
    game = Game(pokemon, secret=secret)
    while game.status == "playing":
        game.guess(solver(pokemon, game.history))
    score = game.guesses_used() if game.status == "won" else FAIL_SCORE
    return score, game

Writing /content/drive/MyDrive/pokemon_wordle/engine.py


## 3d. Tests
You must see `All tests passed`. Then message the group: sample CSV and `feedback()` are ready.

In [ ]:
import importlib, engine, loader
importlib.reload(engine); importlib.reload(loader)
from engine import feedback, Game, play_game
from loader import load_pokemon
import random

mons = load_pokemon()
by_name = {p["name"]: p for p in mons}
print("Loaded", len(mons), "Pokemon")

# 1. The example from the project plan
clue = feedback(by_name["Charizard"], by_name["Gyarados"])
print("Charizard guessed, Gyarados secret ->", clue)
assert clue == ("green", "grey", "green", "up", "up")

# 2. Guessing the secret itself gives all green
assert feedback(by_name["Pikachu"], by_name["Pikachu"]) == ("green",) * 5

# 3. Amber: both types appear in the answer but in the opposite slots
a = {"name": "A", "gen": 1, "type1": "Water", "type2": "Flying", "height": 1.0, "weight": 10.0}
b = {"name": "B", "gen": 3, "type1": "Flying", "type2": "Water", "height": 1.05, "weight": 20.0}
assert feedback(a, b) == ("up", "amber", "amber", "green", "up")

# 4. Game: a solver that always guesses the first unguessed Pokemon
def dumb_solver(pokemon, history):
    seen = {g["name"] for g, _ in history}
    return next(p for p in pokemon if p["name"] not in seen)

score, game = play_game(dumb_solver, mons, by_name[mons[3]["name"]])
print("dumb solver score:", score, "status:", game.status)

# 5. A game cannot go past 8 guesses
g = Game(mons, secret=mons[-1])
for p in mons[:8]:
    g.guess(p)
assert g.status == "lost" and g.guesses_used() == 8
print("All tests passed")

Loaded 20 Pokemon
Charizard guessed, Gyarados secret -> ('green', 'grey', 'green', 'up', 'up')
dumb solver score: 4 status: won
All tests passed


## 4a. Download all 386 Pokémon from PokéAPI
Takes 1 to 2 minutes. Saves to `data/pokemon_full.csv` (the sample stays in place until you check the data).

In [ ]:
import requests, time, csv

RENAME = {
    "nidoran-f": "Nidoran F", "nidoran-m": "Nidoran M", "mr-mime": "Mr. Mime",
    "mime-jr": "Mime Jr.", "ho-oh": "Ho-Oh", "deoxys-normal": "Deoxys",
    "farfetchd": "Farfetch'd",
}

def fetch(pid, tries=3):
    for attempt in range(tries):
        try:
            r = requests.get(f"https://pokeapi.co/api/v2/pokemon/{pid}", timeout=20)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print("retry", pid, e)
            time.sleep(2)
    raise RuntimeError(f"Could not fetch Pokemon {pid}")

def generation(pid):
    return 1 if pid <= 151 else 2 if pid <= 251 else 3

rows = []
for pid in range(1, 387):
    d = fetch(pid)
    types = {t["slot"]: t["type"]["name"].capitalize() for t in d["types"]}
    raw = d["name"]
    name = RENAME.get(raw, raw.capitalize())
    rows.append({
        "name": name,
        "gen": generation(pid),
        "type1": types[1],
        "type2": types.get(2, "None"),
        "height": d["height"] / 10,   # decimetres -> metres
        "weight": d["weight"] / 10,   # hectograms -> kg
    })
    if pid % 50 == 0:
        print("done", pid)
    time.sleep(0.1)

out = f"{DRIVE}/data/pokemon_full.csv"
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["name", "gen", "type1", "type2", "height", "weight"])
    w.writeheader()
    w.writerows(rows)
print("Saved", len(rows), "rows to", out)

done 50
done 100
done 150
done 200
done 250
done 300
done 350
Saved 386 rows to /content/drive/MyDrive/pokemon_wordle/data/pokemon_full.csv


## 4b. Clean and check the data

In [ ]:
import pandas as pd

df = pd.read_csv(f"{DRIVE}/data/pokemon_full.csv", keep_default_na=False)  # keeps the text "None"
print(df.shape)
print(df.head())

assert len(df) == 386, "should be 386 rows"
assert df["name"].is_unique, "duplicate names"
assert df["gen"].value_counts().sort_index().tolist() == [151, 100, 135], "wrong generation counts"
assert (df["height"] > 0).all() and (df["weight"] > 0).all(), "bad height or weight"
assert (df["type1"] != "").all() and (df["type1"] != "None").all(), "every Pokemon needs a type1"
print("Types used:", sorted((set(df["type1"]) | set(df["type2"])) - {"None"}))
print("Pokemon with no second type:", (df["type2"] == "None").sum())
print("Tallest:", df.loc[df["height"].idxmax(), "name"], "| Heaviest:", df.loc[df["weight"].idxmax(), "name"])
print("All checks passed")

(386, 6)
         name  gen  type1   type2  height  weight
0   Bulbasaur    1  Grass  Poison     0.7     6.9
1     Ivysaur    1  Grass  Poison     1.0    13.0
2    Venusaur    1  Grass  Poison     2.0   100.0
3  Charmander    1   Fire    None     0.6     8.5
4  Charmeleon    1   Fire    None     1.1    19.0
Types used: ['Bug', 'Dark', 'Dragon', 'Electric', 'Fairy', 'Fighting', 'Fire', 'Flying', 'Ghost', 'Grass', 'Ground', 'Ice', 'Normal', 'Poison', 'Psychic', 'Rock', 'Steel', 'Water']
Pokemon with no second type: 204
Tallest: Wailord | Heaviest: Groudon
All checks passed


## 4c. Replace the sample with the full file
Only run this after 4b prints `All checks passed`. Then message the group: full CSV is ready.

In [ ]:
import shutil
shutil.copy(f"{DRIVE}/data/pokemon.csv", f"{DRIVE}/data/pokemon_sample_backup.csv")
shutil.copy(f"{DRIVE}/data/pokemon_full.csv", f"{DRIVE}/data/pokemon.csv")

import importlib, loader
importlib.reload(loader)
mons = loader.load_pokemon()
print("Loader now returns", len(mons), "Pokemon")
assert len(mons) == 386

Loader now returns 386 Pokemon


## 4d. Final test on the full data

In [ ]:
import importlib, engine, loader
importlib.reload(engine); importlib.reload(loader)
from engine import feedback, Game, play_game
from loader import load_pokemon
import random

mons = load_pokemon()
by_name = {p["name"]: p for p in mons}
print("Loaded", len(mons), "Pokemon")

# 1. The example from the project plan
clue = feedback(by_name["Charizard"], by_name["Gyarados"])
print("Charizard guessed, Gyarados secret ->", clue)
assert clue == ("green", "grey", "green", "up", "up")

# 2. Guessing the secret itself gives all green
assert feedback(by_name["Pikachu"], by_name["Pikachu"]) == ("green",) * 5

# 3. Amber: both types appear in the answer but in the opposite slots
a = {"name": "A", "gen": 1, "type1": "Water", "type2": "Flying", "height": 1.0, "weight": 10.0}
b = {"name": "B", "gen": 3, "type1": "Flying", "type2": "Water", "height": 1.05, "weight": 20.0}
assert feedback(a, b) == ("up", "amber", "amber", "green", "up")

# 4. Game: a solver that always guesses the first unguessed Pokemon
def dumb_solver(pokemon, history):
    seen = {g["name"] for g, _ in history}
    return next(p for p in pokemon if p["name"] not in seen)

score, game = play_game(dumb_solver, mons, by_name[mons[3]["name"]])
print("dumb solver score:", score, "status:", game.status)

# 5. A game cannot go past 8 guesses
g = Game(mons, secret=mons[-1])
for p in mons[:8]:
    g.guess(p)
assert g.status == "lost" and g.guesses_used() == 8
print("All tests passed")

Loaded 386 Pokemon
Charizard guessed, Gyarados secret -> ('green', 'grey', 'green', 'up', 'up')
dumb solver score: 4 status: won
All tests passed
